In [1]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [2]:
# local embedding model import

MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10225.89it/s]


In [3]:

documents = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="LangChain is used to create AI agents."),
    Document(page_content="LangChain supports retrieval augmented generation."),
    Document(page_content="Chroma stores document embeddings."),
    Document(page_content="FAISS performs fast vector similarity search."),
]

vector_store = Chroma(
    collection_name="mmr_rectriever_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_mmr"
)

vector_store.add_documents(documents)

['29e6bf67-4d6e-45ee-91a0-1b2bb8a73396',
 '20dd3b2f-e36f-4610-aedd-548a2ab33225',
 'ab7b7f2f-8892-4e22-bed6-69bc6d18e285',
 '887eb75f-2613-466a-8654-5a39db53f0f4',
 'bb8d7cae-71ef-42ff-9676-01af98ef01c2']

In [4]:
# MMR retriever তৈরি করছি. MMR মানে Maximum Marginal Relevance

retriever = vector_store.as_retriever(
    search_type="mmr",

    search_kwargs={
        "k": 3,               # k = শেষে কতগুলো document return হবে
        "fetch_k": 10,        # fetch_k = প্রথমে কতগুলো candidate document নিয়ে কাজ করবে
        "lambda_mult": 0.5     # lambda_mult = similarity ও diversity-এর balance. 1 হলে বেশি similarity, 0 হলে বেশি diversity

    }
)

query = "Tell me about LangChain."

results = retriever.invoke(query)


In [5]:
# Result print

print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Query: Tell me about LangChain.
--------------------------------------------------
1. LangChain is used to create AI agents.
2. LangChain helps build LLM applications.
3. FAISS performs fast vector similarity search.
